In [ ]:
import os

# Use only 1 GPU if available
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import pandas as pd  # requires: pip install pandas
import numpy as np
import matplotlib.pyplot as plt
import torch
from chronos import BaseChronosPipeline, Chronos2Pipeline

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

## Peak time forecast

In [ ]:
def mark_near_peak(peak_df: pd.DataFrame,
                                 length: int,
                                 col: str | None = None,
                                 include_peak: bool = False,
                                 return_frame: bool = False,
                                 name: str | None = None) -> pd.Series | pd.DataFrame:
    if length < 1:
        raise ValueError("length must be a positive integer (>=1).")

    s = peak_df[col] if col is not None else peak_df.squeeze("columns")
    s = pd.to_numeric(s, errors="coerce").fillna(0).astype(int)

    arr = s.values.astype(int)
    # Convolve with a ones kernel of width (2*length + 1) to detect any peak in the window
    kernel = np.ones(2 * length + 1, dtype=int)
    count = np.convolve(arr, kernel, mode="same")  # zero-padded at edges

    flags = (count > 0).astype(int)
    if not include_peak:
        flags[arr == 1] = 0  # exclude the peak day itself

    out_name = name or (f"within_pm{length}_of_peak" + ("_incl_peak" if include_peak else ""))
    out = pd.Series(flags, index=s.index, name=out_name)
    return out.to_frame() if return_frame else out

def mask_future_near_peak(near_peak: pd.Series, peaks: pd.Series, start: int, end: int, len_days: int = 3) -> pd.Series:
    """
    near_peak, peaks: full-length series aligned with the same index.
    [start:end): context window (end exclusive).
    len_days: number of days before/after each peak marked as 1 in near_peak.
    """
    ctx = near_peak.iloc[start:end].copy()

    # find all peak indices at or beyond 'end' (future peaks)
    future_peak_idx = np.flatnonzero(peaks.values)[peaks.index[np.flatnonzero(peaks.values)] >= peaks.index[end]]

    if len(future_peak_idx) == 0:
        return ctx  # nothing to mask

    # For each future peak, wipe its ±len_days region within the current window
    for p in future_peak_idx:
        wipe_start = max(start, p - len_days)
        wipe_end   = min(end, p + len_days + 1)
        if wipe_start < end and wipe_end > start:
            # convert to relative window indices
            rel_start = wipe_start - start
            rel_end   = wipe_end - start
            ctx.iloc[rel_start:rel_end] = 0

    return ctx

In [ ]:
states   = ["az","ca","il","md","nj","ny"]
near_len = 3                      # <-- set the window around peaks you want as a covariate
input_len, pred_len, stride = 56, 14, 7
T = len(covid_df)
n_windows = (T - input_len - pred_len) // stride + 1
quantiles = [0.05,0.1,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95]

peaks_df_map = {}
for state in states:
    peaks_df_map[state] = pd.read_csv(f"long observed peak/{state}_long_observed_peak.csv", index_col=0, parse_dates=True)

# Precompute per-state series
per_state_inc   = {st: covid_df[st].astype(float)                             for st in states}  # incidence
per_state_peakS = {st: peaks_df_map[st].squeeze().astype(int)        for st in states} # peak
per_state_near  = {st: mark_near_peak(peaks_df_map[st], near_len, include_peak=True).squeeze().astype(int)  for st in states} # near peak

In [ ]:
per_state_forecast = {st: np.zeros((n_windows, pred_len, len(quantiles)), dtype=np.float32) for st in states}

def _quantile_cols(df, qlevels):
    cols = []
    for q in qlevels:
        if q in df.columns: cols.append(q)
        elif f"{q:g}" in df.columns: cols.append(f"{q:g}")
        elif f"{q}" in df.columns: cols.append(f"{q}")
        else: raise KeyError(f"Quantile column {q} not found in {df.columns.tolist()}")
    return cols

stop = 0

# ------------- windowed predictions -------------
torch.set_num_threads(1)
with torch.inference_mode():
    for w in range(n_windows-stop):
        start = w * stride
        end   = start + input_len       # exclusive context end
        pred_start = covid_df.index[end]
        print(f"[{w+1:02d}/{n_windows}] prediction starts {pred_start:%Y-%m-%d}")

        # assemble long-format batch with TARGET + COVARIATES (past-only)
        frames = []
        for st in states:
            
            # --- UPDATED: Use the near_peak logic instead of d2p ---
            tgt_ctx = mask_future_near_peak(per_state_near[st], per_state_peakS[st], start, end) 
            
            inc_ctx = per_state_inc[st].iloc[start:end]

            frames.append(pd.DataFrame({
                "item_id":   st,
                "timestamp": covid_df.index[start:end],
                "target": tgt_ctx.values,    # TARGET
                "incidence":   inc_ctx.values,     # covariate 1
            }))

        batch = pd.concat(frames, ignore_index=True)

        out = pipeline.predict_df(
            df=batch,
            target="target",                 # target column name
            prediction_length=pred_len,
            quantile_levels=quantiles
            # (optional) future_df=... if you add known-future covariates later
        ).sort_values(["item_id","timestamp"], kind="stable")

        qcols = _quantile_cols(out, quantiles)
        for st in states:
            block = out.loc[out["item_id"] == st, qcols].to_numpy(dtype=np.float32)
            if block.shape != (pred_len, len(quantiles)):
                raise ValueError(f"Unexpected block for {st}: {block.shape}")
            per_state_forecast[st][w, :, :] = block

# Access forecasts
print({k: v.shape for k, v in per_state_forecast.items()})

In [ ]:
# Save the Forecasts 
output_dir = f"../../data/1_raw/{near_len} near peak"
os.makedirs(output_dir, exist_ok=True)

for st in states:
    file_path = os.path.join(output_dir, f"{st}_{near_len}nearpeak.npy")
    np.save(file_path, per_state_forecast[st])
    print(f"Successfully saved {st.upper()} peak forecast to {file_path}")

### test loading one file

In [ ]:
incidence = pd.read_csv("../../data/2_processed/covid_incidence.csv", index_col=0, parse_dates=True)
start_indices = np.array([ 56,  63,  70,  77,  84,  91,  98, 105, 112, 119, 126, 133, 140, 147, 154, 161, 168, 175, 182, 189, 196, 203, 210, 217, 224, 231,
       238, 245, 252, 259, 266, 273, 280, 287, 294, 301, 308, 315, 322,
       329, 336, 343, 350, 357, 364, 371, 378, 385, 392, 399, 406])
state = "nj"

# keep only the median value in axis 3 of the (51, 14, 19) per_state["az"]
probs_1w = per_state_forecast[state][0:-stop, 0:7, 9]
if stop == 0:
       probs_1w = per_state_forecast[state][:, 0:7, 9]
actual_peak = per_state_peakS[state]
actual_peak = actual_peak[actual_peak == 1].index
peak_from_start = (actual_peak - pd.Timestamp("2020-03-15")).days.tolist()

# flatten and plot
# plot observed peak within the length on the graph
plt.figure(figsize=(10, 6))
plt.plot(probs_1w.flatten(), label="Chronos-2 Zero-Shot Forecast", color="blue")
for i in range(len(peak_from_start)):
       plt.axvline(x=peak_from_start[i], color="green", linestyle="--")
plt.title(f"Chronos-2 Zero-Shot Forecast vs Observed Incidence for {state.upper()}")
plt.xlabel("Days since start")
plt.ylabel("Incidence")
plt.legend()
plt.grid()
plt.show()